# Network Intrusion Detection System (NIDS)
**IBM SkillsBuild Cyber Security 2026 — BharatCares**  
**Submitted by:** VIJAY LAXMI SUNIA  
**Project:** Intrusion Detection System  

---

This notebook contains the complete implementation of a rule-based Network Intrusion Detection System built in Python.  
It demonstrates packet capture (live & simulated), a sliding-window detection engine, a thread-safe alert store, and a Flask web dashboard.

### Table of Contents
1. [Project Overview](#1.-Project-Overview)
2. [Install Dependencies](#2.-Install-Dependencies)
3. [Module: AlertStore](#3.-Module:-AlertStore)
4. [Module: Detection Engine](#4.-Module:-Detection-Engine)
5. [Module: Packet Sniffer & Simulator](#5.-Module:-Packet-Sniffer-&-Simulator)
6. [Module: Flask Dashboard](#6.-Module:-Flask-Dashboard)
7. [Main Entry Point](#7.-Main-Entry-Point)
8. [Smoke Test (Simulation)](#8.-Smoke-Test-(Simulation))
9. [Sample Alert Analysis](#9.-Sample-Alert-Analysis)
10. [Detection Rules Summary](#10.-Detection-Rules-Summary)

---
## 1. Project Overview

A **Network Intrusion Detection System (NIDS)** monitors network traffic and raises alerts when suspicious patterns are detected. This implementation:

| Feature | Detail |
|---|---|
| Language | Python 3.9+ |
| Packet capture | Scapy (live) or built-in simulator |
| Detection method | Rule-based, sliding time-window counters |
| Alert storage | Thread-safe in-memory store |
| Dashboard | Flask + vanilla JavaScript |
| Deployment | Single command (`python main.py --simulate`) |

### Attack types detected
- **SYN Flood** — TCP SYN storm targeting a host
- **Port Scan** — Rapid sweep across many destination ports
- **ICMP Flood** — Ping flood / amplification
- **Null Scan** — Stealth scan with all TCP flags cleared
- **Xmas Scan** — Stealth scan with FIN+PSH+URG flags set

### Architecture
```
┌─────────────────────────────────────────────────────┐
│  Daemon Thread (sniffer)                            │
│  Live Scapy capture / Synthetic simulator           │
│         │                                           │
│         ▼                                           │
│  detector.analyze(packet, store)  ◄─ 5 rules        │
│         │                                           │
│         ▼                                           │
│  AlertStore (thread-safe list)                      │
└───────────────────┬─────────────────────────────────┘
                    │  HTTP GET /api/alerts
              ┌─────▼──────┐
              │  Flask App  │  Main thread
              │  port 5000  │
              └─────────────┘
```

---
## 2. Install Dependencies

In [ ]:
# Install required packages
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scapy==2.5.0', 'Flask==3.0.3', '--quiet'])
print('Dependencies installed successfully.')

---
## 3. Module: AlertStore

Thread-safe in-memory store for intrusion alerts.  
Each alert is a dictionary with `timestamp`, `type`, `src_ip`, `dst_ip`, and `detail`.

In [ ]:
"""
nids/alert_store.py
In-memory alert store — thread-safe list of alert dictionaries.
"""

import threading
from datetime import datetime


class AlertStore:
    """Thread-safe, in-memory store for NIDS alerts."""

    def __init__(self):
        self._lock = threading.Lock()
        self._alerts = []

    def add(self, alert_type: str, src_ip: str, dst_ip: str, detail: str = ""):
        """Create and append a new alert; print it to stdout."""
        alert = {
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            "type":      alert_type,
            "src_ip":    src_ip,
            "dst_ip":    dst_ip,
            "detail":    detail,
        }
        with self._lock:
            self._alerts.append(alert)
        print(f"[ALERT] {alert['timestamp']} | {alert_type} | {src_ip} -> {dst_ip} | {detail}")
        return alert

    def get_all(self):
        """Return a snapshot copy of all alerts."""
        with self._lock:
            return list(self._alerts)

    def clear(self):
        """Delete all stored alerts."""
        with self._lock:
            self._alerts.clear()


# Package-level singleton shared across all modules
alert_store = AlertStore()
sniffer_running = True   # toggled by /api/stop and /api/resume

print('AlertStore module loaded.')

---
## 4. Module: Detection Engine

The core analysis component. `analyze(packet, store)` is called for every captured or simulated packet and applies **5 independent detection rules**.

| Rule | Technique | Threshold |
|---|---|---|
| SYN Flood | Sliding-window counter (5 s) | > 100 SYN packets |
| Port Scan | Sliding-window distinct-port set | > 15 unique ports |
| ICMP Flood | Sliding-window counter (5 s) | > 50 ICMP packets |
| Null Scan | Per-packet TCP flag check | flags == 0x00 |
| Xmas Scan | Per-packet TCP flag check | flags & 0x29 == 0x29 |

In [ ]:
"""
nids/detector.py
Detection engine — applies all rules to each packet.
"""

from collections import defaultdict
import time

# ── Sliding-window counters ────────────────────────────────────────────────────
# { src_ip: [(timestamp, dst_port), ...] }
_syn_tracker:       dict = defaultdict(list)
_icmp_tracker:      dict = defaultdict(list)
_port_scan_tracker: dict = defaultdict(set)   # { src_ip: {dst_ports} }

WINDOW               = 5    # seconds
SYN_THRESHOLD        = 100  # SYN packets per WINDOW → SYN Flood
ICMP_THRESHOLD       = 50   # ICMP packets per WINDOW → ICMP Flood
PORT_SCAN_THRESHOLD  = 15   # distinct ports per WINDOW → Port Scan


def _prune(events: list, now: float) -> list:
    """Remove events older than WINDOW seconds."""
    return [e for e in events if now - e[0] < WINDOW]


# ── Scapy layer shims (allow notebook to run without live Scapy import) ────────
try:
    from scapy.all import IP, TCP, UDP, ICMP as _scapy_ICMP
    _IP, _TCP, _UDP, _ICMP = IP, TCP, UDP, _scapy_ICMP
except ImportError:
    # Minimal stubs so the detector code is importable for testing
    class _LayerStub:
        __name__ = 'Stub'
    _IP = type('IP',  (), {'__name__': 'IP'})()
    _TCP = type('TCP', (), {'__name__': 'TCP'})()
    _UDP = type('UDP', (), {'__name__': 'UDP'})()
    _ICMP = type('ICMP', (), {'__name__': 'ICMP'})()


def analyze(packet, store):
    """Analyse a single packet against all detection rules."""
    if _IP not in packet:
        return

    src = packet[_IP].src
    dst = packet[_IP].dst
    now = time.time()

    # ── Rule 1: SYN Flood ──────────────────────────────────────────────────────
    if _TCP in packet and packet[_TCP].flags == 0x02:  # SYN only
        _syn_tracker[src] = _prune(_syn_tracker[src], now)
        _syn_tracker[src].append((now, packet[_TCP].dport))
        if len(_syn_tracker[src]) > SYN_THRESHOLD:
            store.add("SYN Flood", src, dst,
                      f"{len(_syn_tracker[src])} SYNs in {WINDOW}s")
            _syn_tracker[src].clear()

    # ── Rule 2: Port Scan ──────────────────────────────────────────────────────
    if _TCP in packet or _UDP in packet:
        proto = _TCP if _TCP in packet else _UDP
        dport = packet[proto].dport
        if src not in _port_scan_tracker:
            _port_scan_tracker[src] = {}
        _port_scan_tracker[src][dport] = now
        # Prune old ports
        _port_scan_tracker[src] = {
            p: t for p, t in _port_scan_tracker[src].items() if now - t < WINDOW
        }
        if len(_port_scan_tracker[src]) > PORT_SCAN_THRESHOLD:
            store.add("Port Scan", src, dst,
                      f"{len(_port_scan_tracker[src])} distinct ports in {WINDOW}s")
            _port_scan_tracker[src].clear()

    # ── Rule 3: ICMP Flood ─────────────────────────────────────────────────────
    if _ICMP in packet:
        _icmp_tracker[src] = _prune(_icmp_tracker[src], now)
        _icmp_tracker[src].append((now, 0))
        if len(_icmp_tracker[src]) > ICMP_THRESHOLD:
            store.add("ICMP Flood", src, dst,
                      f"{len(_icmp_tracker[src])} ICMP pkts in {WINDOW}s")
            _icmp_tracker[src].clear()

    # ── Rule 4: Null Scan (all TCP flags = 0) ─────────────────────────────────
    if _TCP in packet and packet[_TCP].flags == 0x00:
        store.add("Null Scan", src, dst, f"dport={packet[_TCP].dport}")

    # ── Rule 5: Xmas Scan (FIN+PSH+URG) ──────────────────────────────────────
    if _TCP in packet and (packet[_TCP].flags & 0x29) == 0x29:
        store.add("Xmas Scan", src, dst, f"dport={packet[_TCP].dport}")


print('Detection engine loaded.')
print(f'  SYN Flood threshold : {SYN_THRESHOLD} SYNs / {WINDOW}s')
print(f'  Port Scan threshold : {PORT_SCAN_THRESHOLD} ports / {WINDOW}s')
print(f'  ICMP Flood threshold: {ICMP_THRESHOLD} pkts / {WINDOW}s')

---
## 5. Module: Packet Sniffer & Simulator

Provides two operating modes via a single `start()` function:

- **Live mode** — calls `scapy.sniff()` on a real NIC. Requires Npcap on Windows.
- **Simulation mode** — generates synthetic `_FakePacket` objects that mirror the Scapy API, so the detector runs identically in both modes. No drivers or root required.

In [ ]:
"""
nids/sniffer.py
Packet sniffer — two modes:
  1. Live capture via Scapy (requires Npcap on Windows).
  2. Simulation mode — generates realistic synthetic packets.
"""

import time
import random
import threading

# ── Simulation data ────────────────────────────────────────────────────────────
_ATTACKER_IPS = [
    "192.168.1.101", "10.0.0.45",   "172.16.0.22",
    "203.0.113.77",  "198.51.100.5", "192.0.2.88",
]
_VICTIM_IPS    = ["192.168.1.1", "10.0.0.1", "172.16.0.1"]
_COMMON_PORTS  = [22, 23, 80, 443, 3306, 5432, 8080, 8443, 21, 25, 110, 3389]

# Attack scenarios: (name, weight)
_SCENARIOS = [
    ("syn_flood",  20),
    ("port_scan",  20),
    ("icmp_flood", 15),
    ("null_scan",   5),
    ("xmas_scan",   5),
    ("normal",     35),
]
_WEIGHTS = [w for _, w in _SCENARIOS]
_NAMES   = [n for n, _ in _SCENARIOS]


# ── Fake packet classes (mirror Scapy's packet API) ───────────────────────────

class _IPLayer:
    def __init__(self, src, dst):
        self.src = src
        self.dst = dst

class _TransportLayer:
    def __init__(self, sport, dport, flags):
        self.sport = sport
        self.dport = dport
        self.flags = flags

class _ICMPLayer:
    pass

class _FakePacket:
    """Minimal packet object that mimics Scapy's packet interface."""

    def __init__(self, src, dst, proto="TCP", sport=None, dport=None,
                 tcp_flags=0x02, icmp=False):
        self._layers   = {"IP": True}
        self.src       = src
        self.dst       = dst
        self.proto     = proto
        self.sport     = sport or random.randint(1024, 65535)
        self.dport     = dport or random.choice(_COMMON_PORTS)
        self.tcp_flags = tcp_flags
        self._icmp     = icmp

    def __contains__(self, layer_cls):
        name = getattr(layer_cls, "__name__", str(layer_cls))
        if name == "IP":   return True
        if name == "TCP":  return self.proto == "TCP"
        if name == "UDP":  return self.proto == "UDP"
        if name == "ICMP": return self._icmp
        return False

    def __getitem__(self, layer_cls):
        name = getattr(layer_cls, "__name__", str(layer_cls))
        if name == "IP":            return _IPLayer(self.src, self.dst)
        if name in ("TCP", "UDP"): return _TransportLayer(self.sport, self.dport, self.tcp_flags)
        if name == "ICMP":          return _ICMPLayer()
        raise KeyError(name)


# ── Packet factory ─────────────────────────────────────────────────────────────

def _make_packet(scenario: str) -> _FakePacket:
    src = random.choice(_ATTACKER_IPS)
    dst = random.choice(_VICTIM_IPS)

    if scenario == "syn_flood":  return _FakePacket(src, dst, proto="TCP", tcp_flags=0x02)
    if scenario == "port_scan":  return _FakePacket(src, dst, proto="TCP",
                                                     dport=random.randint(1, 65535), tcp_flags=0x02)
    if scenario == "icmp_flood": return _FakePacket(src, dst, proto="ICMP", icmp=True)
    if scenario == "null_scan":  return _FakePacket(src, dst, proto="TCP", tcp_flags=0x00)
    if scenario == "xmas_scan":  return _FakePacket(src, dst, proto="TCP", tcp_flags=0x29)
    # normal — mix of TCP/UDP
    proto = random.choice(["TCP", "UDP"])
    return _FakePacket(src, dst, proto=proto, dport=random.choice(_COMMON_PORTS), tcp_flags=0x18)


# ── Simulation runner ──────────────────────────────────────────────────────────

def _simulate(store, running_flag: list, rate: float = 0.05):
    """
    Generate synthetic packets at `rate` seconds/packet (~20 pkt/s).
    Bursts every ~10 s to trigger detection rules.
    Stops when running_flag[0] is False.
    """
    print("[*] Running in SIMULATION mode — generating synthetic traffic...")
    t0 = time.time()

    while running_flag[0]:
        elapsed = time.time() - t0
        if int(elapsed) % 10 == 0 and int(elapsed) > 0:
            scenario = random.choices(_NAMES, weights=_WEIGHTS, k=1)[0]
            burst    = 120 if "flood" in scenario else 20
            for _ in range(burst):
                if not running_flag[0]: break
                analyze(_make_packet(scenario), store)
            time.sleep(1)
        else:
            scenario = random.choices(_NAMES, weights=_WEIGHTS, k=1)[0]
            analyze(_make_packet(scenario), store)
            time.sleep(rate)


# ── Live capture ───────────────────────────────────────────────────────────────

def _live(store, interface=None):
    """Capture real packets with Scapy."""
    try:
        from scapy.all import sniff as scapy_sniff
    except ImportError:
        print("[!] Scapy not installed — falling back to simulation mode.")
        _simulate(store, [True])
        return

    def _cb(packet):
        analyze(packet, store)

    print(f"[*] Live capture on: {interface or 'default'}")
    try:
        scapy_sniff(iface=interface, prn=_cb, store=False, count=0)
    except Exception as e:
        print(f"[!] Scapy capture failed: {e} — falling back to simulation.")
        _simulate(store, [True])


def start(store, interface=None, simulate=False, running_flag=None):
    """Start the sniffer. Pass running_flag=[True] to allow external stop."""
    rf = running_flag if running_flag is not None else [True]
    if simulate:
        _simulate(store, rf)
    else:
        _live(store, interface)


print('Sniffer module loaded.')

---
## 6. Module: Flask Dashboard

A lightweight Flask application that:
- Serves `index.html` — a dark-themed real-time dashboard
- Exposes a REST JSON API for alert management and sniffer control

| Endpoint | Method | Purpose |
|---|---|---|
| `/` | GET | Serve the HTML dashboard |
| `/api/alerts` | GET | Return all alerts as JSON |
| `/api/alerts/clear` | POST | Clear all alerts |
| `/api/status` | GET | Sniffer running state |
| `/api/stop` | POST | Pause the sniffer |
| `/api/resume` | POST | Resume the sniffer |

In [ ]:
"""
dashboard/app.py
Flask web dashboard for the NIDS.
"""

# NOTE: Run this cell only to view the source. To launch the full app,
# run Cell 7 (Main Entry Point) which starts Flask in a background thread.

DASHBOARD_SOURCE = '''
from flask import Flask, render_template, jsonify

def create_app(store, running_flag):
    app = Flask(__name__, template_folder="templates")

    @app.route("/")
    def index():
        return render_template("index.html")

    @app.route("/api/alerts")
    def api_alerts():
        return jsonify(store.get_all())

    @app.route("/api/alerts/clear", methods=["POST"])
    def api_clear():
        store.clear()
        return jsonify({"status": "cleared"})

    @app.route("/api/status")
    def api_status():
        return jsonify({"running": running_flag[0]})

    @app.route("/api/stop", methods=["POST"])
    def api_stop():
        running_flag[0] = False
        return jsonify({"status": "stopped"})

    @app.route("/api/resume", methods=["POST"])
    def api_resume():
        running_flag[0] = True
        return jsonify({"status": "running"})

    return app
'''

print('Flask dashboard source:')
print(DASHBOARD_SOURCE)

---
## 7. Main Entry Point

Wires all modules together: launches the sniffer in a daemon thread, then starts the Flask dashboard.

**To run from the command line:**
```bash
# Simulation mode (no Npcap required)
python main.py --simulate

# Live capture on Wi-Fi
python main.py -i Wi-Fi

# Custom port
python main.py --simulate --port 8080
```
Then open **http://127.0.0.1:5000** in your browser.

In [ ]:
"""
main.py
Main entry point — starts the sniffer in a background thread
and launches the Flask dashboard.
"""

import threading
import argparse
import sys
import os

# NOTE: In the installed project, the imports below resolve from the
# nids_project/ package. The code is shown here for reference.

MAIN_SOURCE = '''
import threading, argparse, sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from nids import sniffer
from nids import alert_store, sniffer_running
from dashboard.app import app

def main():
    parser = argparse.ArgumentParser(description="Network Intrusion Detection System")
    parser.add_argument("-i", "--interface", default="Wi-Fi",
                        help="Network interface to sniff (default: Wi-Fi).")
    parser.add_argument("--port", type=int, default=5000,
                        help="Dashboard port (default: 5000)")
    parser.add_argument("--simulate", action="store_true", default=False,
                        help="Run in simulation mode — no Npcap/root required.")
    args = parser.parse_args()

    sniff_thread = threading.Thread(
        target=sniffer.start,
        kwargs={"interface": args.interface, "simulate": args.simulate},
        daemon=True,
        name="sniffer",
    )
    sniff_thread.start()

    mode = "SIMULATION" if args.simulate else f"LIVE ({args.interface})"
    print(f"[*] Mode     : {mode}")
    print(f"[*] Dashboard: http://127.0.0.1:{args.port}")

    app.run(host="0.0.0.0", port=args.port, debug=False, use_reloader=False)

if __name__ == "__main__":
    main()
'''

print(MAIN_SOURCE)

---
## 8. Smoke Test (Simulation)

Runs the simulator for **15 seconds** and prints a breakdown of all alerts generated.  
Expected result: multiple alert types across SYN Flood, Port Scan, ICMP Flood, Null Scan, and Xmas Scan.

In [ ]:
import threading
import time

# Fresh store and flag for this test run
test_store       = AlertStore()
test_running     = [True]

# Run simulation in a daemon thread for 15 seconds
sim_thread = threading.Thread(
    target=_simulate,
    args=(test_store, test_running),
    daemon=True,
    name="smoke-sim"
)
sim_thread.start()

print("[*] Smoke test running for 15 seconds...")
time.sleep(15)
test_running[0] = False   # signal the thread to stop
time.sleep(0.5)

# ── Results ───────────────────────────────────────────────────────────────────
alerts = test_store.get_all()
print(f"\n{'='*50}")
print(f" SMOKE TEST RESULTS")
print(f"{'='*50}")
print(f" Total alerts : {len(alerts)}")

counts = {}
for a in alerts:
    counts[a['type']] = counts.get(a['type'], 0) + 1

for k, v in sorted(counts.items()):
    print(f"   {k:<20} {v}")

print()
if len(alerts) > 0:
    print(" RESULT: PASS ✓ — alerts generated successfully")
else:
    print(" RESULT: FAIL ✗ — no alerts generated")

---
## 9. Sample Alert Analysis

After the smoke test, analyse the captured alerts — counts by type, top attacking IPs, and a timeline summary.

In [ ]:
from collections import Counter

alerts = test_store.get_all()

# ── 1. Alerts by type ─────────────────────────────────────────────────────────
type_counter = Counter(a['type'] for a in alerts)
print("Alerts by type:")
for t, c in type_counter.most_common():
    bar = '█' * c
    print(f"  {t:<20} {c:>4}  {bar}")

print()

# ── 2. Top attacking source IPs ───────────────────────────────────────────────
src_counter = Counter(a['src_ip'] for a in alerts)
print("Top attacking source IPs:")
for ip, c in src_counter.most_common(5):
    print(f"  {ip:<20} {c} alerts")

print()

# ── 3. Most targeted destination IPs ─────────────────────────────────────────
dst_counter = Counter(a['dst_ip'] for a in alerts)
print("Most targeted destination IPs:")
for ip, c in dst_counter.most_common(3):
    print(f"  {ip:<20} {c} alerts")

print()

# ── 4. Sample alert records ───────────────────────────────────────────────────
print("Sample alerts (last 5):")
print(f"  {'Timestamp':<22} {'Type':<15} {'Src IP':<18} {'Dst IP':<15} Detail")
print("  " + "-"*90)
for a in alerts[-5:]:
    print(f"  {a['timestamp']:<22} {a['type']:<15} {a['src_ip']:<18} {a['dst_ip']:<15} {a['detail']}")

---
## 10. Detection Rules Summary

A consolidated reference for all five detection rules implemented in this NIDS.

In [ ]:
rules = [
    {
        "Rule":        "SYN Flood",
        "Protocol":    "TCP",
        "Technique":   "Sliding-window counter",
        "Window":      "5 s",
        "Threshold":   f">{SYN_THRESHOLD} SYN packets",
        "Flag/Check":  "TCP flags == 0x02 (SYN only)",
        "Description": "Overwhelms a host's TCP connection table",
    },
    {
        "Rule":        "Port Scan",
        "Protocol":    "TCP/UDP",
        "Technique":   "Sliding-window distinct-port set",
        "Window":      "5 s",
        "Threshold":   f">{PORT_SCAN_THRESHOLD} unique ports",
        "Flag/Check":  "Track unique dst ports per src IP",
        "Description": "Reconnaissance sweep to find open services",
    },
    {
        "Rule":        "ICMP Flood",
        "Protocol":    "ICMP",
        "Technique":   "Sliding-window counter",
        "Window":      "5 s",
        "Threshold":   f">{ICMP_THRESHOLD} ICMP packets",
        "Flag/Check":  "ICMP layer present",
        "Description": "Ping flood / amplification attack",
    },
    {
        "Rule":        "Null Scan",
        "Protocol":    "TCP",
        "Technique":   "Per-packet flag check",
        "Window":      "N/A",
        "Threshold":   "Any single packet",
        "Flag/Check":  "TCP flags == 0x00 (all clear)",
        "Description": "Stealth probe to bypass naive firewalls",
    },
    {
        "Rule":        "Xmas Scan",
        "Protocol":    "TCP",
        "Technique":   "Per-packet flag check",
        "Window":      "N/A",
        "Threshold":   "Any single packet",
        "Flag/Check":  "TCP flags & 0x29 == 0x29 (FIN+PSH+URG)",
        "Description": "Named for lit-up flag bits; bypasses some stateless ACLs",
    },
]

# Pretty-print as a table
header = f"{'Rule':<15} {'Proto':<8} {'Threshold':<28} {'Flag/Check':<35} Description"
print(header)
print("-" * len(header))
for r in rules:
    print(f"{r['Rule']:<15} {r['Protocol']:<8} {r['Threshold']:<28} {r['Flag/Check']:<35} {r['Description']}")

print()
print(f"Shared sliding window : {WINDOW} seconds")
print("All constants defined in detector.py — tunable without logic changes.")

---
## Project Summary

| Item | Value |
|---|---|
| **Submitted by** | VIJAY LAXMI SUNIA |
| **Programme** | IBM SkillsBuild Cyber Security 2026 — BharatCares |
| **Project** | Network Intrusion Detection System (NIDS) |
| **Language** | Python 3.9+ |
| **Dependencies** | scapy==2.5.0, Flask==3.0.3 |
| **Detection rules** | 5 (SYN Flood, Port Scan, ICMP Flood, Null Scan, Xmas Scan) |
| **Dashboard** | Flask + vanilla JS, real-time polling, dark theme |
| **Simulation** | Built-in synthetic traffic generator — no hardware required |

---
*End of VIJAYLAXMISUNIA_INTRUSIONDETECTIONSYSTEM.ipynb*